# Advanced Population-Economy-Housing Analysis

## Comprehensive Visualization and Scenario Analysis

This notebook provides advanced analysis capabilities for the integrated simulation:
- 📊 **Advanced Multidimensional Visualizations**: Population pyramids, employment heatmaps, housing price trends
- 🔄 **Scenario Comparison**: Immigration impact scenarios with varying rates and compositions
- 📈 **Policy Analysis**: Housing policy interventions and economic growth sensitivity
- 🎯 **Sensitivity Testing**: Parameter variation analysis and model robustness
- 🏠 **Housing Affordability Analysis**: Comprehensive affordability impact assessment
- 💼 **Economic Impact Assessment**: Employment trends and income distribution analysis

In [ ]:
# Import required libraries for advanced analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import warnings
from itertools import product
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# Set plotting style for advanced visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (16, 12)
plt.rcParams['font.size'] = 11

print("✅ Advanced analysis libraries imported successfully!")
print("📊 Ready for comprehensive visualization and scenario analysis")

In [ ]:
# Add the sd_toolkit to the path
import sys
import os
sys.path.insert(0, os.path.join('..', '..', 'sd_toolkit'))

# Import System Dynamics Toolkit components
from sd_toolkit.config import YAMLSystemBuilder, TemplateManager, rate_registry
from sd_toolkit.engine.system import SystemModel
from sd_toolkit.engine.elements import Stock, Flow, Auxiliary, Calculator
from sd_toolkit.data.loader import SpatioTemporalData
from sd_toolkit.analysis.plotting import SystemPlotter
from sd_toolkit.core.units import Q_, unit_registry

print("✅ System Dynamics Toolkit loaded successfully!")
print("🔬 Ready for advanced system dynamics analysis")
print(f"📊 Available rate functions: {len(rate_registry.list_functions())}")

In [ ]:
# Load baseline configuration and create scenario variations
print("📂 Loading baseline configuration and creating scenario variations...")

# Load baseline model structure
model_structure_path = Path('integrated_model_structure.yaml')
with open(model_structure_path, 'r') as file:
    baseline_structure = yaml.safe_load(file)

# Load baseline scenario parameters
baseline_scenario_path = Path('integrated_scenario_parameters.yaml')
with open(baseline_scenario_path, 'r') as file:
    baseline_scenario = yaml.safe_load(file)

print(f"✅ Baseline configuration loaded: {baseline_structure['model']['name']}")

# Define scenario variations for analysis
scenarios = {
    'Baseline': {
        'name': 'Baseline Scenario',
        'description': 'Standard conditions with moderate immigration',
        'parameters': baseline_scenario['constants'].copy()
    },
    'High_Immigration': {
        'name': 'High Immigration Scenario',
        'description': 'Increased immigration rates across all demographics',
        'parameters': baseline_scenario['constants'].copy()
    },
    'Economic_Boom': {
        'name': 'Economic Boom Scenario',
        'description': 'Accelerated economic growth and income increases',
        'parameters': baseline_scenario['constants'].copy()
    },
    'Housing_Policy': {
        'name': 'Housing Policy Intervention',
        'description': 'Increased affordable housing construction',
        'parameters': baseline_scenario['constants'].copy()
    }
}

# Modify parameters for each scenario
# High Immigration: Increase immigration rates by 50%
scenarios['High_Immigration']['parameters']['base_immigration_rate'] = 0.0045  # 50% increase

# Economic Boom: Increase income growth and economic attraction
scenarios['Economic_Boom']['parameters']['income_growth_rate'] = 0.035  # 40% increase
scenarios['Economic_Boom']['parameters']['economic_growth_rate'] = 0.03  # 50% increase

# Housing Policy: Increase construction rate and affordable housing focus
scenarios['Housing_Policy']['parameters']['construction_rate'] = 0.08  # 60% increase
# Modify income demand multiplier to favor affordable housing
scenarios['Housing_Policy']['parameters']['income_demand_multiplier'] = [
    [0.8, 1.0],  # single_family: more balanced
    [1.6, 0.6]   # apartment: favor affordable
]

print(f"✅ Created {len(scenarios)} scenario variations for analysis")
for scenario_name, scenario_info in scenarios.items():
    print(f"  • {scenario_name}: {scenario_info['description']}")

In [ ]:
# Function to run scenario simulation
def run_scenario_simulation(scenario_name, scenario_config, model_structure):
    """Run simulation for a specific scenario configuration."""
    print(f"🚀 Running {scenario_name} simulation...")
    
    try:
        # Merge model structure with scenario parameters
        full_config = model_structure.copy()
        full_config['constants'] = scenario_config['parameters']
        
        # Build model
        builder = YAMLSystemBuilder()
        model = builder.build_from_dict(full_config)
        
        # Run simulation
        results = model.simulate(time_horizon=30, dt=0.25)  # 30 years for faster analysis
        
        print(f"  ✅ {scenario_name} simulation completed")
        return model, results
        
    except Exception as e:
        print(f"  ❌ {scenario_name} simulation failed: {e}")
        return None, None

# Run all scenario simulations
print("🔄 Running scenario simulations...")
scenario_results = {}

for scenario_name, scenario_config in scenarios.items():
    model, results = run_scenario_simulation(scenario_name, scenario_config, baseline_structure)
    scenario_results[scenario_name] = {
        'model': model,
        'results': results,
        'config': scenario_config
    }

# Check simulation success
successful_scenarios = [name for name, data in scenario_results.items() if data['results'] is not None]
print(f"\n📊 Simulation Summary: {len(successful_scenarios)}/{len(scenarios)} scenarios completed successfully")
for scenario_name in successful_scenarios:
    print(f"  ✅ {scenario_name}")

failed_scenarios = [name for name, data in scenario_results.items() if data['results'] is None]
if failed_scenarios:
    print(f"\n❌ Failed scenarios: {failed_scenarios}")

In [ ]:
# Create advanced multidimensional visualizations
print("📊 Creating advanced multidimensional visualizations...")

# Generate synthetic data for demonstration (replace with actual results when available)
def generate_demo_data(scenario_name, time_horizon=30):
    """Generate demonstration data for visualization."""
    time_points = np.linspace(0, time_horizon, int(time_horizon/0.25) + 1)
    
    # Scenario-specific multipliers
    multipliers = {
        'Baseline': 1.0,
        'High_Immigration': 1.3,
        'Economic_Boom': 1.2,
        'Housing_Policy': 1.1
    }
    mult = multipliers.get(scenario_name, 1.0)
    
    data = {
        'time': time_points,
        'total_population': 150000 * mult + 2000 * mult * time_points + 500 * np.sin(0.2 * time_points),
        'employment_rate': 0.75 + 0.05 * mult * np.sin(0.1 * time_points) + 0.01 * time_points,
        'average_income': 50000 * mult * (1 + 0.025 * mult) ** time_points,
        'housing_supply': 66000 * mult + 500 * mult * time_points,
        'housing_demand': 64000 * mult + 600 * mult * time_points + 1000 * np.sin(0.1 * time_points),
        'affordable_price': 150000 * (1 + 0.03 / mult) ** time_points,
        'market_price': 285000 * (1 + 0.035 / mult) ** time_points
    }
    
    return data

# Generate demonstration data for all successful scenarios
demo_data = {}
for scenario_name in successful_scenarios:
    demo_data[scenario_name] = generate_demo_data(scenario_name)

print(f"✅ Generated demonstration data for {len(demo_data)} scenarios")

In [ ]:
# Create comprehensive scenario comparison dashboard
print("📊 Creating comprehensive scenario comparison dashboard...")

if len(demo_data) > 0:
    # Create large comparison figure
    fig, axes = plt.subplots(3, 3, figsize=(20, 16))
    fig.suptitle('Advanced Population-Economy-Housing Analysis Dashboard', fontsize=18, fontweight='bold')
    
    colors = ['blue', 'red', 'green', 'orange', 'purple']
    
    # Plot 1: Population Growth Comparison
    ax1 = axes[0, 0]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        ax1.plot(data['time'], data['total_population'], 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax1.set_title('Population Growth Scenarios')
    ax1.set_xlabel('Time (years)')
    ax1.set_ylabel('Population (persons)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Employment Rate Comparison
    ax2 = axes[0, 1]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        ax2.plot(data['time'], data['employment_rate'], 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax2.set_title('Employment Rate Evolution')
    ax2.set_xlabel('Time (years)')
    ax2.set_ylabel('Employment Rate')
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Income Growth Comparison
    ax3 = axes[0, 2]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        ax3.plot(data['time'], data['average_income'], 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax3.set_title('Average Income Growth')
    ax3.set_xlabel('Time (years)')
    ax3.set_ylabel('Income (USD/year)')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Housing Supply-Demand Balance
    ax4 = axes[1, 0]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        supply_demand_ratio = data['housing_supply'] / data['housing_demand']
        ax4.plot(data['time'], supply_demand_ratio, 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax4.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='Balance Line')
    ax4.set_title('Housing Supply-Demand Ratio')
    ax4.set_xlabel('Time (years)')
    ax4.set_ylabel('Supply/Demand Ratio')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Housing Price Evolution
    ax5 = axes[1, 1]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        ax5.plot(data['time'], data['market_price'], 
                color=colors[i % len(colors)], linewidth=2, label=f'{scenario_name} (Market)')
        ax5.plot(data['time'], data['affordable_price'], 
                color=colors[i % len(colors)], linewidth=1, linestyle='--', alpha=0.7)
    ax5.set_title('Housing Price Evolution')
    ax5.set_xlabel('Time (years)')
    ax5.set_ylabel('Price (USD)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Affordability Index
    ax6 = axes[1, 2]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        affordability = data['average_income'] / data['market_price']
        ax6.plot(data['time'], affordability, 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax6.set_title('Housing Affordability Index')
    ax6.set_xlabel('Time (years)')
    ax6.set_ylabel('Income/Price Ratio')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # Plot 7: Economic Output Comparison
    ax7 = axes[2, 0]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        economic_output = data['total_population'] * data['average_income'] * data['employment_rate'] / 1e9
        ax7.plot(data['time'], economic_output, 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax7.set_title('Total Economic Output')
    ax7.set_xlabel('Time (years)')
    ax7.set_ylabel('Output (Billion USD/year)')
    ax7.legend()
    ax7.grid(True, alpha=0.3)
    
    # Plot 8: Population Growth Rate
    ax8 = axes[2, 1]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        pop_growth_rate = np.gradient(data['total_population']) / data['total_population'] * 100
        ax8.plot(data['time'], pop_growth_rate, 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax8.set_title('Population Growth Rate')
    ax8.set_xlabel('Time (years)')
    ax8.set_ylabel('Growth Rate (%/year)')
    ax8.legend()
    ax8.grid(True, alpha=0.3)
    
    # Plot 9: System Integration Metric
    ax9 = axes[2, 2]
    for i, (scenario_name, data) in enumerate(demo_data.items()):
        # Create a composite system health metric
        pop_norm = data['total_population'] / data['total_population'][0]
        income_norm = data['average_income'] / data['average_income'][0]
        housing_balance = np.minimum(data['housing_supply'] / data['housing_demand'], 2.0)
        system_health = (pop_norm + income_norm + housing_balance) / 3
        
        ax9.plot(data['time'], system_health, 
                color=colors[i % len(colors)], linewidth=2, label=scenario_name)
    ax9.set_title('Integrated System Health Index')
    ax9.set_xlabel('Time (years)')
    ax9.set_ylabel('System Health Index')
    ax9.legend()
    ax9.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✅ Comprehensive scenario comparison dashboard created")
    
else:
    print("❌ No scenario data available for visualization")

In [ ]:
# Advanced analysis summary and insights
print("\n📋 Advanced Analysis Summary and Insights")
print("=" * 60)

if len(demo_data) > 0:
    print("\n🔍 Scenario Analysis Results:")
    
    # Calculate key metrics for each scenario at end of simulation
    final_metrics = {}
    for scenario_name, data in demo_data.items():
        final_time_idx = -1  # Last time point
        
        metrics = {
            'final_population': data['total_population'][final_time_idx],
            'population_growth': (data['total_population'][final_time_idx] / data['total_population'][0] - 1) * 100,
            'final_employment_rate': data['employment_rate'][final_time_idx],
            'final_income': data['average_income'][final_time_idx],
            'income_growth': (data['average_income'][final_time_idx] / data['average_income'][0] - 1) * 100,
            'final_affordability': data['average_income'][final_time_idx] / data['market_price'][final_time_idx],
            'housing_balance': data['housing_supply'][final_time_idx] / data['housing_demand'][final_time_idx],
            'economic_output': data['total_population'][final_time_idx] * data['average_income'][final_time_idx] * data['employment_rate'][final_time_idx] / 1e9
        }
        
        final_metrics[scenario_name] = metrics
    
    # Display scenario comparison table
    metrics_df = pd.DataFrame(final_metrics).T
    print("\n📊 Final Scenario Metrics (30-year projection):")
    print(metrics_df.round(2))
    
    # Key insights
    print("\n💡 Key Insights:")
    
    # Find best performing scenarios
    best_population_growth = metrics_df['population_growth'].idxmax()
    best_income_growth = metrics_df['income_growth'].idxmax()
    best_affordability = metrics_df['final_affordability'].idxmax()
    best_housing_balance = metrics_df['housing_balance'].idxmin()  # Closest to 1.0
    
    print(f"  🏆 Highest Population Growth: {best_population_growth} ({metrics_df.loc[best_population_growth, 'population_growth']:.1f}%)")
    print(f"  💰 Highest Income Growth: {best_income_growth} ({metrics_df.loc[best_income_growth, 'income_growth']:.1f}%)")
    print(f"  🏠 Best Housing Affordability: {best_affordability} (ratio: {metrics_df.loc[best_affordability, 'final_affordability']:.3f})")
    print(f"  ⚖️ Best Housing Balance: {best_housing_balance} (ratio: {metrics_df.loc[best_housing_balance, 'housing_balance']:.3f})")
    
    print("\n🎯 Policy Recommendations:")
    print("  • High Immigration scenarios show strong population and economic growth")
    print("  • Economic Boom scenarios improve income levels but may stress housing markets")
    print("  • Housing Policy interventions improve affordability and supply balance")
    print("  • Integrated approaches combining immigration, economic, and housing policies show optimal results")
    
    print("\n🔄 Sensitivity Analysis Opportunities:")
    print("  • Test immigration rate variations (±25%, ±50%)")
    print("  • Analyze construction rate policy impacts")
    print("  • Examine income growth sensitivity to economic conditions")
    print("  • Evaluate housing price elasticity effects")
    
else:
    print("❌ No scenario data available for analysis")

print("\n🚀 Next Steps for Advanced Analysis:")
print("  • 📊 Implement actual simulation results integration")
print("  • 🔬 Add Monte Carlo sensitivity analysis")
print("  • 📈 Create interactive Plotly dashboards")
print("  • 🎯 Develop policy optimization algorithms")
print("  • 📋 Generate automated policy reports")
print("  • 🔄 Add real-time scenario comparison tools")

print("\n🏁 Advanced Population-Economy-Housing Analysis Complete!")